In [0]:
"""
07_execution_events.py

Creates the Silver Execution Events table.

Input:
    parsed_events

Output:
    execution_events

Author:
Sumanth Vempalle

Version:
2.1.0
"""

from pyspark import pipelines as dp

from pyspark.sql.functions import col


# ============================================================
# Execution Events
# ============================================================

@dp.table(
    name="execution_events",
    comment="Validated manufacturing execution events.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dp.expect_or_drop(
    "valid_execution_id",
    "execution_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_work_order_id",
    "work_order_id IS NOT NULL",
)

@dp.expect_or_drop(
    "valid_product_code",
    "product_code IS NOT NULL",
)

@dp.expect(
    "positive_quantity",
    "quantity > 0",
)

@dp.expect_or_drop(
    "valid_production_line",
    "production_line IS NOT NULL",
)

def execution_events():

    df = dp.read_stream("parsed_events")

    return (

        df

        # -----------------------------------------
        # Keep only execution events
        # -----------------------------------------

        .filter(
            col("event_type") == "EXECUTION_STARTED"
        )

        # -----------------------------------------
        # Business columns
        # -----------------------------------------

        .select(

            "event_id",
            "event_timestamp",
            "event_version",

            "plant_code",

            "execution_id",
            "work_order_id",

            "product_code",

            "source_system",
            "correlation_id",

            "bronze_ingestion_timestamp",
            "silver_processing_timestamp",

            "payload.sap_order_number",

            "payload.planned_shift",

            "payload.quantity",

            "payload.status",

            "payload.production_line",

        )

    )